# FX Pipeline: Data Download, Tidy Formatting, and Feature Engineering

This notebook downloads FX data, converts it to tidy form, and generates the feature set
used to build `fx_features_wide.csv`.

## Table of Contents
1. [Setup](#setup)
2. [Select FX Pairs](#select-fx-pairs)
3. [Download & Tidy Data](#download--tidy-data)
4. [Feature Engineering](#feature-engineering)
5. [Wide Matrix & Correlation Features](#wide-matrix--correlation-features)
6. [PCA Factors](#pca-factors)
7. [Save Final Dataset](#save-final-dataset)
8. [Next Steps: ML & PnL (Placeholders)](#next-steps-ml--pnl-placeholders)


## 1. Setup <a id='setup'></a>
Installs dependencies and imports core libraries.

In [1]:
# ============================================================
# 0. Setup
# ============================================================

!pip install yfinance -q
!pip install pandas numpy scikit-learn -q

import pandas as pd
import numpy as np
import yfinance as yf
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ============================================================
# 1. Select FX Pairs (tightest spreads)
# ============================================================

pairs = [
    "EURUSD=X", "USDJPY=X", "GBPUSD=X", "AUDUSD=X", "NZDUSD=X",
    "USDCHF=X", "USDCAD=X", "EURJPY=X", "GBPJPY=X", "AUDJPY=X",
    "EURGBP=X", "EURNZD=X", "EURCHF=X", "CHFJPY=X", "CADJPY=X",
    "GBPCHF=X", "GBPCAD=X", "AUDNZD=X", "AUDCAD=X", "AUDCHF=X",
    "NZDJPY=X", "NZDCAD=X", "NZDCHF=X", "CADCHF=X", "EURCAD=X"
]

print("Downloading:", len(pairs), "pairs")

# ============================================================
# 2. Download last 1000 trading days in tidy format
# ============================================================

raw = yf.download(
    tickers=pairs,
    period="1000d",
    interval="1d",
    group_by="ticker",
    auto_adjust=True,
    progress=False
)

# Convert to tidy
rows = []
for p in pairs:
    dfp = raw[p].reset_index().dropna()
    dfp["pair"] = p
    rows.append(dfp)

tidy = pd.concat(rows, ignore_index=True)

tidy = tidy.rename(columns={"index": "Date"})
tidy["Date"] = pd.to_datetime(tidy["Date"], utc=True)
tidy = tidy[["Date", "pair", "Open", "High", "Low", "Close"]]

print("Tidy shape:", tidy.shape)
display(tidy.head())

# ============================================================
# 3. Compute all features
# ============================================================

df = tidy.sort_values(["pair", "Date"]).copy()

# Log prices
df["log_close"] = np.log(df["Close"])

# Daily log returns
df["log_ret"] = df.groupby("pair")["log_close"].diff()

# 30-day rolling volatility
df["vol30"] = df.groupby("pair")["log_ret"].rolling(30).std().reset_index(level=0, drop=True)

# 5-day and 10-day momentum
df["mom5"] = df.groupby("pair")["log_close"].diff(5)
df["mom10"] = df.groupby("pair")["log_close"].diff(10)

# ============================================================
# 4. Convert to wide format
# ============================================================

# Pivot returns, vol, momentum
wide_ret   = df.pivot(index="Date", columns="pair", values="log_ret").add_prefix("ret_")
wide_vol30 = df.pivot(index="Date", columns="pair", values="vol30").add_prefix("vol30_")
wide_mom5  = df.pivot(index="Date", columns="pair", values="mom5").add_prefix("mom5_")
wide_mom10 = df.pivot(index="Date", columns="pair", values="mom10").add_prefix("mom10_")

# Align all dates
wide = pd.concat([wide_ret, wide_vol30, wide_mom5, wide_mom10], axis=1)
wide = wide.sort_index().dropna(how="any")

print("Wide matrix initial shape:", wide.shape)

# ============================================================
# 5. Compute 60-day rolling cross-pair correlation features
# ============================================================

corr_window = 60
corr_features = []

for i in range(len(wide)):
    if i < corr_window:
        corr_features.append([np.nan] * len(wide_ret.columns))
        continue

    window_data = wide_ret.iloc[i - corr_window : i]
    corr_mat = window_data.corr()

    # remove diagonal
    np.fill_diagonal(corr_mat.values, np.nan)

    avg_corr = corr_mat.mean(axis=1).values
    corr_features.append(avg_corr)

corr_df = pd.DataFrame(
    corr_features,
    index=wide.index,
    columns=[f"corr60_{c.replace('ret_','')}" for c in wide_ret.columns]
)

wide = pd.concat([wide, corr_df], axis=1)
wide = wide.dropna()

print("After adding correlation features:", wide.shape)

# ============================================================
# 6. PCA on returns
# ============================================================

scale = StandardScaler()
ret_scaled = scale.fit_transform(wide_ret.loc[wide.index].values)

pca = PCA(n_components=3)
pc = pca.fit_transform(ret_scaled)

wide["PC1"] = pc[:, 0]
wide["PC2"] = pc[:, 1]
wide["PC3"] = pc[:, 2]

print("Explained variance:", pca.explained_variance_ratio_)

# ============================================================
# 7. Save final wide feature matrix
# ============================================================

wide_out = wide.reset_index()
wide_out.to_csv("fx_features_wide.csv", index=False)

print("\nSaved fx_features_wide.csv with shape:", wide_out.shape)
display(wide_out.head())


Downloading: 25 pairs
Tidy shape: (24923, 6)


Price,Date,pair,Open,High,Low,Close
0,2022-01-25 00:00:00+00:00,EURUSD=X,1.132387,1.132503,1.126684,1.132413
1,2022-01-26 00:00:00+00:00,EURUSD=X,1.130352,1.131350,1.127205,1.130454
2,2022-01-27 00:00:00+00:00,EURUSD=X,1.124227,1.124480,1.113350,1.124354
3,2022-01-28 00:00:00+00:00,EURUSD=X,1.114616,1.117006,1.112335,1.114703
4,2022-01-31 00:00:00+00:00,EURUSD=X,1.115138,1.121692,1.114504,1.115237


Wide matrix initial shape: (966, 100)
After adding correlation features: (906, 125)
Explained variance: [0.30404442 0.24307138 0.15441521]

Saved fx_features_wide.csv with shape: (906, 129)


,Date,ret_AUDCAD=X,ret_AUDCHF=X,ret_AUDJPY=X,ret_AUDNZD=X,ret_AUDUSD=X,ret_CADCHF=X,ret_CADJPY=X,ret_CHFJPY=X,ret_EURCAD=X,...,corr60_NZDCAD=X,corr60_NZDCHF=X,corr60_NZDJPY=X,corr60_NZDUSD=X,corr60_USDCAD=X,corr60_USDCHF=X,corr60_USDJPY=X,PC1,PC2,PC3
0,2022-05-31 00:00:00+00:00,-0.000839,0.005863,0.008740,0.001407,0.004512,0.005802,0.009334,0.003441,-0.001510,...,0.086059,0.208094,0.310715,0.152836,-0.178922,-0.022089,0.192371,2.679692,2.491809,0.445658
1,2022-06-01 00:00:00+00:00,-0.002761,-0.000321,0.005716,0.004280,-0.001750,0.002730,0.008707,0.006037,-0.004882,...,0.089187,0.208506,0.310586,0.154223,-0.176591,-0.024222,0.189526,-1.015207,3.091782,0.781752
2,2022-06-02 00:00:00+00:00,0.000947,0.002320,0.009441,0.004548,-0.001256,0.001435,0.008496,0.007099,-0.005640,...,0.096376,0.236745,0.343181,0.155913,-0.170068,0.015724,0.235007,-1.127546,3.709939,0.021759
3,2022-06-03 00:00:00+00:00,0.004776,0.006479,0.011314,0.000794,0.013211,0.001853,0.006095,0.004262,0.001748,...,0.099687,0.233842,0.342638,0.151216,-0.153003,0.019388,0.233299,5.323363,-0.016458,1.270230
4,2022-06-06 00:00:00+00:00,-0.006821,-0.002834,-0.002397,-0.000848,-0.008854,0.003774,0.004894,0.001083,-0.001222,...,0.094604,0.230933,0.338293,0.144749,-0.145911,0.029488,0.238442,-2.636461,2.606439,0.025017


## 8. Next Steps: ML & PnL (Placeholders) <a id='next-steps-ml--pnl-placeholders'></a>

Use additional sections below for your modeling and PnL evaluation code.


### 8.1 ML Ranking Model
Add your LightGBM (or other) ranking model pipeline here.

In [2]:
# TODO: Paste your ML ranking pipeline code here


### 8.2 PnL Evaluation & Plots
Add your PnL evaluation logic and plotting code (strategy vs benchmarks, per-pair PnL, etc.).

In [3]:
# TODO: Paste your PnL evaluation and plotting code here
